# 代码目录结构

主要由 `utils`，`schemes` 和 `EXPERIMENTs` 三个目录组成：
+ `utils` 中都是一些基本算法的实现，比如 `tools.py` 中包含必要的 PRF，Hash 算法，解析数据库的算法，`TSet.py` 和 `REMM.py` 分别是 SPX 和我们所提加密数据库的 EMM 算法，`TDAG` 是范围查询的 SRC 算法的实现。
+ `schemes` 是建立在 `utils` 基础上对 SPX 和我们的 FinEDB 方案的实现，这两个方案分别记录在 `spx.py` 和 `fdb.py` 中。
+ `EXPERIMENTs` 中记录实验使用的代码，实验结果写入相应的 `csv` 文件中。

# 简单使用示例

In [1]:
from schemes import fdb  # 导入 fdb 模块
ct_fdb = fdb.Client()  # 创建 client 对象
ct_fdb.load_tables("./data/sf0.001", ["customer"])  # 加载原始数据内容，这里选择的是 sf0.001 目录下的 customer 表

--------------------LOAD COMPLETE--------------------


加载原始数据之后，`ct_fdb.Raw_Tables` 中记录了每张表的信息，这些信息包括表名，内容，以及每个属性对应的类型。

In [2]:
table_info = ct_fdb.Raw_Tables[0]  # 将第一张表（也就是 customer 表）的内容传给 table_info
(t_name, t_data, t_type) = table_info  # customer 表的信息

In [3]:
t_name

'customer'

In [4]:
t_data

,_id,C_NAME,C_ADDRESS,C_NATIONKEY,C_PHONE,C_ACCTBAL,MKT_SEGMENT,C_COMMENT
0,1,Customer#000000001,"IVhzIApeRb ot,c,E",15,25-989-741-2988,711.56,BUILDING,"to the even, regular platelets. regular, ironi..."
1,2,Customer#000000002,"XSTf4,NCwDVaWNe6tEgvwfmRchLXak",13,23-768-687-3665,121.65,AUTOMOBILE,l accounts. blithely ironic theodolites integr...
2,3,Customer#000000003,MG9kdTD2WBHm,1,11-719-748-3364,7498.12,AUTOMOBILE,"deposits eat slyly ironic, even instructions...."
3,4,Customer#000000004,XxVSJsLAGtn,4,14-128-190-5944,2866.83,MACHINERY,"requests. final, regular ideas sleep final accou"
4,5,Customer#000000005,KvpyuHCplrB84WgAiGV6sYpZq7Tj,3,13-750-942-6364,794.47,HOUSEHOLD,n accounts will have to unwind. foxes cajole a...
...,...,...,...,...,...,...,...,...
145,146,Customer#000000146,"GdxkdXG9u7iyI1,,y5tq4ZyrcEy",3,13-835-723-3223,3328.68,FURNITURE,ffily regular dinos are slyly unusual requests...
146,147,Customer#000000147,"6VvIwbVdmcsMzuu,C84GtBWPaipGfi7DV",18,28-803-187-4335,8071.40,AUTOMOBILE,ress packages above the blithely regular packa...
147,148,Customer#000000148,BhSPlEWGvIJyT9swk vCWE,11,21-562-498-6636,2135.60,HOUSEHOLD,ing to the carefully ironic requests. carefull...
148,149,Customer#000000149,3byTHCp2mNLPigUrrq,19,29-797-439-6760,8959.65,AUTOMOBILE,al instructions haggle against the slyly bold w


In [5]:
t_type

[1, 1, 1, 1, 1, 1, 1, 0]

这里主要注意一下 `t_type`，我们规定 0 为不会出现在 Where 语句中的属性，1 为可以执行相等判断的属性，2 为可以执行相等和范围查询的属性，3 为可以执行Join的属性。
当然这里我们只是为了实验方便做的简单约束，所有的 type 都是在 `tool.py` 中进行了预定义。至于又需要执行范围又需要执行Join的属性，对于实验来说是不必要的，并不纳入其中。

加载完数据之后，我们便可以生成 `EMM`，这里直接调用 `construct_index` 方法：

In [6]:
emm = ct_fdb.construct_index()

与 SPX 不同，我们只使用一个单独的 EMM 来表示所有的表和关系，每一行数据我们都采用一个 Binary Fuse filter (BFF) 来存储，而这些所有的 BFFs 也都存储在 EMM 中。

构建服务器对象，只需要传入 `emm` 即可：

In [7]:
sv_fdb = fdb.Server(emm)

生成 tokens 的过程：我们一共生成三个 tokens，tk1 相当于 stag，tk2 代表 Where 语句中的判断，tk3 代表 Select 语句中的属性。
这里以 Select C_NAME, MKT_SGEMENT From customer Where C_NATIONKEY == 3 为例：

In [8]:
from collections import namedtuple
SQL_Query = namedtuple("SQL_Query", ["Select", "Where", "Value"])  # 规定查询的格式
select_att = ["C_NAME", "MKT_SEGMENT"]
where_att = ["C_NATIONKEY"]
value_list = ["3"]
query_tuple = SQL_Query(select_att, where_att, value_list)

In [9]:
tk = ct_fdb.gen_token(query_tuple, "customer")  # 根据输入的查询，生成对应的 token
[f"tk{x+1}: {y}" for x, y in enumerate(tk)]

["tk1: b'\\x85\\xc0\\xe8\\xa6~\\t^+Y\\xb9I\\xb7\\xd7\\x02\\xef\\\\\\xf4\\x1b\\x94\\xbf\\xfa\\xc9\\x06\\x0bat/\\x86\\x02G\\xa7E'",
 "tk2: (b'\\x11\\x18\\x03\\x04\\x02\\x80\\x00', b'\\xcd~\\xc6\\xe2')",
 "tk3: [b'\\x11\\x18\\x03\\x01 \\x02\\x00', b'\\x11\\x18\\x03\\x04\\x10\\x01\\x00']"]

服务器查询时，也只需要使用对应的 token 即可，得到结果为一系列 AES 密文：

In [10]:
enc_res = sv_fdb.query(tk)
enc_res

[[[b'{"IV": "1CWmrSU9pHKIuTPM6IctTQ==", "CipherText": "Q9ZTv2JTEqrZOIpEqJ+/+6kafJU6lcY/x1MD+mGoiv8="}'],
  [b'{"IV": "5lhGTd7rHPExaW3bwMaPeQ==", "CipherText": "ZkJZr+0EHSCwmUWv67hE7NsYVGqQcKuQGkwV8hhJ5Fs="}'],
  [b'{"IV": "SjhbRn1t9GlFRFBjHn/kMw==", "CipherText": "9hPEGGHqfIzP/anTKrZoxONB8ESHTEexSI8dAfC7YoA="}'],
  [b'{"IV": "JtPZ00uoVlM8uEwglT4IFw==", "CipherText": "5P+mEOP78gXjAr41TYlRn1THFurXcUUCA7OQaClQWGM="}'],
  [b'{"IV": "RXir007q4925wl9LOZgUsA==", "CipherText": "YctMLPOG6SsWYwjTq1NlFvFXh59MGRedtYz5VLtEjoU="}'],
  [b'{"IV": "arGyNExPXLu+IPWndWDTQw==", "CipherText": "euihFwmpeU29OGGihmZd1YlBUU3b8kUThNSPDHUNhfc="}'],
  [b'{"IV": "gpxQq2wTlR4dIKgBuKeBJw==", "CipherText": "2tkaRYZqJKLH3JvsVKzCI6EAqkLEV3lap+V+9IhbokY="}'],
  [b'{"IV": "+gEcwTavSmARLIGtuxJDQA==", "CipherText": "bzNGY2LcD1KvH5Ypm+KojarUpVJHz/P1uzeSTPQ5aPk="}'],
  [b'{"IV": "N8P4i96tMzLbQz/hWZRN+w==", "CipherText": "/kPgoOOktK4r/2VSl2PPvRSSflvjrjSEQ4cY/S2YgzQ="}']],
 [[b'{"IV": "OsA/ZoWhGLL/GWgYc+VU5w==", "CipherText": 

客户端再对结果进行解密，可以得到所有 select 语句中 C_NAME 和 MKT_SEGMENT 对应的值：

In [11]:
ct_fdb.decrypt_res(enc_res, "customer")

[[b'Customer#000000005',
  b'Customer#000000013',
  b'Customer#000000022',
  b'Customer#000000023',
  b'Customer#000000027',
  b'Customer#000000040',
  b'Customer#000000064',
  b'Customer#000000122',
  b'Customer#000000146'],
 [b'HOUSEHOLD',
  b'BUILDING',
  b'MACHINERY',
  b'HOUSEHOLD',
  b'BUILDING',
  b'BUILDING',
  b'BUILDING',
  b'HOUSEHOLD',
  b'FURNITURE']]

这里的查询可以是 conjunction 形式，也就是说 Where 语句中的条件可以是一个列表，比如 Select C_NAME From customer Where C_NATIONKEY == 3 and MKT_SEGMENT == HOUSEHOLD：

In [12]:
select_att = ["C_NAME"]
where_att = ["C_NATIONKEY", "MKT_SEGMENT"]
value_list = ["3", "HOUSEHOLD"]
query_tuple = SQL_Query(select_att, where_att, value_list)

tk = ct_fdb.gen_token(query_tuple, "customer")
enc_res = sv_fdb.query(tk)
ct_fdb.decrypt_res(enc_res, "customer")

[[b'Customer#000000005', b'Customer#000000023', b'Customer#000000122']]